# 5.0 Do the recipe and conditions affect the final material properties?
EDA of the material topic, before the baseline maths (5.1) and the models (5.2). The property used here is strength (stress at 10% strain), the one with enough data; other properties such as acoustic or fire resistance can be added later if time and data allow.

A teaching notebook on the property data for the **Material strength** model. It follows the workflow in `MycoMaterials/workflow.md`, but on our own table, and with the tools from class: a heatmap, scikit-learn, and a decision tree.

**The question.** We have many candidate inputs (species, substrate, particle size, spawn %, temperature, water %, C:N, pressing, density). Which of them can our data actually *learn* something about?

**The catch.** The stress rows come from only 5 labs (`dataset_id`), and only 4 of them also report density. Rows from the same lab share a machine, a protocol and a person, like replicate wells on one plate. So the honest unit of evidence is the **lab**, not the row. Every test below leaves one whole lab out and predicts it from the other four (*leave-one-lab-out*).

Steps:
1. Missing values and how many different values each input has
2. Does the input change *within* a lab? (if not, it is tangled with the lab)
3. Heatmaps: correlation between inputs, and lab vs recipe (confounding)
4. Baselines: the number to beat
5. Forward selection by hand: add one input at a time, keep it only if the held-out error falls
6. A shallow decision tree, and permutation importance on held-out labs
7. A mixed model with lab as a random effect

Run from the top (Run All). Nothing is written to disk.

In [ ]:
### Cell 1. Imports and the property table
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.compose import make_column_transformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree
from mycodemo.config import INTERIM_DATA_DIR
from mycodemo.property_data import build_property_table, load_tables

EXTRACTED = INTERIM_DATA_DIR / "extracted"
tables = load_tables()
t = build_property_table(tables)

# Two inputs the property table does not carry yet: water % (fermentation.csv) and C:N (substrate.csv)
link = tables["sample"][["sample_id", "fermentation_id", "substrate_id"]]
ferm = pd.read_csv(EXTRACTED / "fermentation.csv", encoding="utf-8-sig")[["fermentation_id", "water_pct_wt"]]
subs = pd.read_csv(EXTRACTED / "substrate.csv", encoding="utf-8-sig")[["substrate_id", "cn_estimate"]]
extra = link.merge(ferm, on="fermentation_id", how="left").merge(subs, on="substrate_id", how="left")
t = t.merge(extra[["sample_id", "water_pct_wt", "cn_estimate"]], on="sample_id", how="left")

# Stress rows only; uncoated, as in the Material strength model
d = t[t["stress_mpa"].notna() & ~t["coated"]].copy()
d["log_stress"] = np.log10(d["stress_mpa"])
d["log_density"] = np.log10(d["density_kg_m3"])
print(f"{len(d)} stress rows, {d['n_specimens'].sum():.0f} blocks, {d['dataset_id'].nunique()} labs")
d["dataset_id"].value_counts()

### Step 1. Missing values and distinct values
A column that is mostly empty, or has the same value everywhere, cannot teach the model anything, like a housekeeping gene that is expressed identically in every sample.

In [ ]:
### Cell 2. Missing share and number of distinct values per candidate input
CANDIDATES = ["species_name", "substrate_group", "particle_size_class", "spawn_pct_wt",
              "incubation_temp_c", "water_pct_wt", "cn_estimate", "total_growth_days",
              "pre_compressed", "compacted_after_growth", "log_density"]
screen = pd.DataFrame({
    "missing_%": (d[CANDIDATES].isna().mean() * 100).round(0),
    "distinct_values": d[CANDIDATES].nunique(),
})
screen

### Step 2. Does the input change within a lab?
This is the most important table in the notebook. If an input only changes *between* labs, then "this substrate" and "this lab" always come together, and no model can tell them apart. It is like an experiment where every treatment was run on a different day: the treatment effect and the day effect are the same column.

`labs_varying_it` counts the labs in which the input takes more than one value. An input with 0 can only be learned from differences between labs, which is the weakest kind of evidence we have.

In [ ]:
### Cell 3. Within-lab variation
within = d.groupby("dataset_id")[CANDIDATES].nunique(dropna=True).T
screen["labs_with_data"] = d.groupby("dataset_id")[CANDIDATES].apply(lambda g: g.notna().any()).sum()
screen["labs_varying_it"] = (within > 1).sum(axis=1)
screen["verdict"] = np.where(screen["labs_varying_it"] >= 2, "learnable within labs",
                    np.where(screen["labs_varying_it"] == 1, "one lab only", "between labs only"))
display(within)
screen

### Step 3. Heatmaps
**Left: correlation between numeric inputs.** Spearman (rank) correlation, so one extreme value cannot dominate. Two inputs close to +1 or −1 carry the same information (collinear); keep the one that is easier to measure or explain.

**Right: lab vs substrate.** Each cell counts stress rows. If each lab sits in its own column, substrate and lab are confounded.

In [ ]:
### Cell 4. Correlation heatmap and lab-by-recipe heatmap
numeric = ["log_stress", "log_density", "particle_size_class", "spawn_pct_wt", "incubation_temp_c",
           "water_pct_wt", "cn_estimate", "total_growth_days", "pre_compressed", "compacted_after_growth"]
corr = d[numeric].corr(method="spearman")

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax[0])
ax[0].set_title("Spearman correlation (rows pooled across labs)")
sns.heatmap(pd.crosstab(d["dataset_id"], d["substrate_group"]), annot=True, fmt="d", cmap="Greens", ax=ax[1])
ax[1].set_title("Stress rows per lab and substrate group")
plt.tight_layout(); plt.show()
pd.crosstab(d["dataset_id"], d["species_name"])

**How to read it.** A strong colour in the left map, pooled over all labs, can be a lab difference in disguise. For example, if one lab grew at 28 °C and made weak blocks for other reasons, temperature will look "important". The within-lab table (Step 2) is the check on every colour in this map.

### Step 4. Baselines: the number to beat
Any model must beat two simple guesses when a whole lab is hidden:
- **mean**: the average log stress of the other labs, for every block
- **density only**: a straight line from log density to log stress, fitted on the other labs

Score: the **fold error**, 10^(mean absolute error in log10). A fold error of 2.0 means the guess is typically off by a factor of 2 (half or double). Each lab gets its own score, then the labs are averaged, so the big labs do not outweigh the small ones. Rows are weighted by `n_specimens` (a mean row of 5 blocks counts 5×).

Note that density is only known *after* the block is grown and dried. It is fine for understanding strength, but a grower planning a recipe does not have it yet. That is why the **Material strength** model predicts density from the recipe first (two steps).

In [ ]:
### Cell 5. Leave-one-lab-out scorer and the two baselines
CATEGORICAL = {"species_name", "substrate_group"}

def make_linear(features):
    cat = [f for f in features if f in CATEGORICAL]
    num = [f for f in features if f not in CATEGORICAL]
    parts = []
    if cat:
        parts.append((OneHotEncoder(handle_unknown="ignore"), cat))
    if num:
        parts.append(("passthrough", num))
    return make_pipeline(make_column_transformer(*parts), LinearRegression())

def lolo(features, data, model_fn=make_linear):
    # Leave one lab out; returns the fold error per held-out lab
    rows = data.dropna(subset=list(features) + ["log_stress"])
    out = {}
    for train, test in LeaveOneGroupOut().split(rows, groups=rows["dataset_id"]):
        tr, te = rows.iloc[train], rows.iloc[test]
        if not features:
            pred = np.full(len(te), np.average(tr["log_stress"], weights=tr["n_specimens"]))
        else:
            m = model_fn(features)
            m.fit(tr[features], tr["log_stress"], **{f"{m.steps[-1][0]}__sample_weight": tr["n_specimens"]})
            pred = m.predict(te[features])
        err = np.average(np.abs(pred - te["log_stress"]), weights=te["n_specimens"])
        out[te["dataset_id"].iloc[0]] = 10 ** err
    return pd.Series(out)

# Use the same rows for every comparison: rows with density (needed for the baseline)
base_rows = d[d["log_density"].notna()].copy()
baselines = pd.DataFrame({"mean": lolo([], base_rows), "density_only": lolo(["log_density"], base_rows)})
baselines.loc["average over labs"] = baselines.mean()
baselines.round(2)

### Step 5. Forward selection by hand
Start from density only. Try adding each other input on its own and re-run the leave-one-lab-out test. Keep an input only if the average fold error goes **down**. With 5 labs, a change in the second decimal is noise; look for a clear drop that holds in most labs.

This is the cheap, honest cousin of RFE (recursive feature elimination). RFE would rank inputs by how much the model used them on the training data; here every decision is made on a lab the model has not seen. Rows missing the added input are dropped, so the baseline is re-scored on the same rows each time (`baseline_same_rows`). An input that only one lab varies (Step 2) can still look helpful here by shifting that lab's level; treat a gain that comes from one lab with suspicion.

In [ ]:
### Cell 6. Add one input at a time to density
rows = []
for f in [c for c in CANDIDATES if c != "log_density"]:
    sub = base_rows.dropna(subset=[f])
    if sub["dataset_id"].nunique() < 3:
        rows.append({"added": f, "labs": sub["dataset_id"].nunique(), "note": "fewer than 3 labs, skipped"})
        continue
    with_f = lolo(["log_density", f], sub)
    without = lolo(["log_density"], sub)
    rows.append({"added": f, "labs": sub["dataset_id"].nunique(),
                 "baseline_same_rows": round(without.mean(), 2),
                 "with_input": round(min(with_f.mean(), 99), 2),
                 "labs_improved": int((with_f < without).sum()),
                 # A held-out lab outside the training range makes a straight line extrapolate wildly
                 "note": "extrapolated wildly (shown as 99)" if with_f.mean() > 99 else ""})
forward = pd.DataFrame(rows).sort_values("with_input")
forward

### Step 5b. SelectKBest, the tool from class
`SelectKBest` (in `sklearn.feature_selection`) scores each input against stress **on its own**, using all rows pooled, and keeps the top *k*. It is the "filter" step in `workflow.md`, like ranking genes by fold change before looking at pathways.
- `f_regression`: an F-test for a straight-line link (a high F and a small p mean a strong link)
- `mutual_info_regression`: how much knowing the input reduces uncertainty about stress; it also catches curved links (0 = no information)

**Compare it with the forward-selection table above.** SelectKBest sees every row, including the lab it is "predicting", and cannot tell a recipe effect from a lab fingerprint. An input that ranks high here but made the held-out-lab error *worse* in Step 5 is most likely marking which lab a row came from.

In [ ]:
### Cell 6b. SelectKBest: the one-input-at-a-time ranking from class
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

# SelectKBest needs numbers with no gaps: one-hot the two category columns, fill gaps with the column median
kb_num = ["log_density", "particle_size_class", "spawn_pct_wt", "incubation_temp_c", "water_pct_wt",
          "cn_estimate", "total_growth_days", "pre_compressed"]
X = pd.get_dummies(base_rows[["species_name", "substrate_group"]], dtype=float)
X = pd.concat([base_rows[kb_num].fillna(base_rows[kb_num].median()), X], axis=1)
y = base_rows["log_stress"]

kb = SelectKBest(score_func=f_regression, k=5).fit(X, y)
mi = mutual_info_regression(X, y, random_state=0)
ranking = pd.DataFrame({
    "F_score": kb.scores_.round(1),
    "p_value": kb.pvalues_,
    "mutual_info": mi.round(3),
    "in_top_5": kb.get_support(),
}, index=X.columns).sort_values("F_score", ascending=False)
ranking["p_value"] = ranking["p_value"].map("{:.1g}".format)
ranking

### Step 6. A shallow decision tree
A decision tree asks yes/no questions, like a dichotomous key in a field guide: "density above 200 kg/m³? then is it pressed?". Each answer splits the blocks into two groups with more similar strength. That makes it easy to read, which is why class starts with it.

**Why only depth 2.** A deep tree keeps splitting until each leaf holds a handful of rows, and with our data that means it memorises labs. We cap it at 2 questions and at least 15 rows per leaf.

**Two kinds of importance.**
- `feature_importances_` (built into the tree) measures how much each input helped *on the training data*. It flatters inputs with many distinct values, and it rewards lab fingerprints.
- **Permutation importance** shuffles one input in the *held-out* lab and measures how much worse the prediction gets, like a knockout experiment done on new animals. Only this one says whether the input helps on a lab the model has not seen.

In [ ]:
### Cell 7. Fit one tree on all labs and draw it
# Numeric inputs that most density labs report (a tree drops rows with gaps, so spawn % and temperature would leave 2 labs)
TREE_FEATURES = ["log_density", "pre_compressed", "cn_estimate", "total_growth_days"]
tree_rows = base_rows.dropna(subset=TREE_FEATURES)
tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=15, random_state=0)
tree.fit(tree_rows[TREE_FEATURES], tree_rows["log_stress"], sample_weight=tree_rows["n_specimens"])

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=TREE_FEATURES, filled=True, rounded=True, precision=2)
plt.title("Depth-2 tree for log10 stress (value = mean log10 stress in that group)")
plt.show()
print(f"{len(tree_rows)} rows from {tree_rows['dataset_id'].nunique()} labs")
pd.Series(tree.feature_importances_, index=TREE_FEATURES).round(2).sort_values(ascending=False)

In [ ]:
### Cell 8. Tree vs baselines on held-out labs, and permutation importance on the held-out lab
def make_tree(features):
    return make_pipeline(DecisionTreeRegressor(max_depth=2, min_samples_leaf=15, random_state=0))

compare = pd.DataFrame({
    "mean": lolo([], tree_rows),
    "density_only": lolo(["log_density"], tree_rows),
    "tree_depth2": lolo(TREE_FEATURES, tree_rows, model_fn=make_tree),
})
compare.loc["average over labs"] = compare.mean()
display(compare.round(2))

perm = []
for train, test in LeaveOneGroupOut().split(tree_rows, groups=tree_rows["dataset_id"]):
    tr, te = tree_rows.iloc[train], tree_rows.iloc[test]
    m = DecisionTreeRegressor(max_depth=2, min_samples_leaf=15, random_state=0)
    m.fit(tr[TREE_FEATURES], tr["log_stress"], sample_weight=tr["n_specimens"])
    r = permutation_importance(m, te[TREE_FEATURES], te["log_stress"], n_repeats=30,
                               random_state=0, scoring="neg_mean_absolute_error")
    perm.append(pd.Series(r.importances_mean, index=TREE_FEATURES, name=te["dataset_id"].iloc[0]))
perm = pd.DataFrame(perm)
perm.loc["average over labs"] = perm.mean()
print("Permutation importance on the held-out lab (increase in log10 error when shuffled; ~0 = not used or not helpful)")
perm.round(3)

### Step 7. Mixed model: lab as a random effect
A mixed model says: "each lab has its own baseline strength (a random offset), and on top of that, density and the other inputs act the same way in every lab". Biologically, it is like fitting a dose-response across litters with litter as a random effect. The slopes it reports are therefore learned mostly from variation **within** labs, which is the evidence we trust.

Add an input to `formula` only if it survived Steps 2 and 5. Read the coefficient with its 95% interval: if the interval crosses 0, the data cannot say which way the input works.

In [ ]:
### Cell 9. Mixed model with lab as random effect
formula = "log_stress ~ log_density + pre_compressed"   # edit after reading Steps 2 and 5
mm_rows = base_rows.dropna(subset=["log_density", "pre_compressed"])
mixed = smf.mixedlm(formula, mm_rows, groups=mm_rows["dataset_id"]).fit()
print(mixed.summary())

### Training history
A learning curve for the two small models above (density only, and the depth-2 tree): each lab is held out once and the model is trained on 1, 2, ... other whole labs (a lab is never split), 5 random draws each. `train_score` and `val_score` are the fold error (a factor, 1.0 is perfect, lower is better) on the training labs and on the hidden lab. A curve that does not fall with more labs says the labs, not the model, are the limit. The mixed model (Step 7) is a single fit, so it has no history. CSV and PNG go to `Desktop/MycoMaterials/kaggle/history/`.

In [ ]:
import datetime
from mycodemo.config import HISTORY_DIR
from mycodemo.modeling.curves import grouped_learning_curve
from mycodemo.plot_history import save_history


def fold_error_scorer(features, model_fn):
    def fit_score(train, test):
        m = model_fn(features)
        m.fit(train[features], train["log_stress"], **{f"{m.steps[-1][0]}__sample_weight": train["n_specimens"]})
        err = np.average(np.abs(m.predict(test[features]) - test["log_stress"]), weights=test["n_specimens"])
        return min(10 ** err, 99)   # a straight line from one lab can extrapolate wildly (as in Step 5)
    return fit_score


for label, features, data, model_fn in [
        ("density_only_linear", ["log_density"], base_rows, make_linear),
        ("tree_depth2", TREE_FEATURES, tree_rows, make_tree)]:
    rows_ok = data.dropna(subset=features + ["log_stress"])
    hist = grouped_learning_curve(rows_ok, "dataset_id", fold_error_scorer(features, model_fn))
    csv, png = save_history(
        hist, HISTORY_DIR, "leaveonelabout", model="material_strength", run=f"screening_{label}",
        notebook="5.0.feature_screening_material", train_set="other labs, whole labs added 1, 2, ...",
        test_set="the left-out lab", date=datetime.date.today().isoformat(),
        settings={"features": features, "cv": "leave one lab out", "repeats": 5,
                  "score": "fold error, x factor (lower is better)"})
    print(csv, png, sep="\n")
    display(hist.round(2))

### What to take away
- The number of labs (5), not the number of rows (171), limits how many inputs we can afford: realistically 1 or 2 beyond density.
- Inputs that only change between labs (Step 2) stay out of the model. They can still be used in the app as **Warnings** with literature evidence.
- The heatmap, the tree's own importances and pooled correlations are *where to look*, not proof. Only held-out-lab scores (Steps 4, 5, 6) count as proof.
- Random forests, SHAP and PCA are left out on purpose: with 5 labs they would mostly measure lab fingerprints.